In [1]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np

DATA_DIR = '/content/drive/MyDrive/MIRAGE/data/raw/GSE98320'

clean_df = pd.read_csv(f'{DATA_DIR}/gse98320_master_pathology_labels.csv')
gene_expr_matrix = pd.read_parquet(f'{DATA_DIR}/gse98320_gene_expression.parquet')

print("Master table shape:", clean_df.shape)
print("Gene expression shape:", gene_expr_matrix.shape)

clean_df.head()

Mounted at /content/drive
Master table shape: (1208, 19)
Gene expression shape: (18835, 1208)


,GSM,mirage_label,g,cg,i,ci,t,ct,v,cv,ah,ptc,c4d_ordinal,ifta_ordinal,in i-ifta set n=234?,archetype cluster,d96,mmdx,split_group
0,GSM2590943,TCMR,0.0,0.0,2.0,3.0,3.0,3.0,0.0,1.0,NaN,0.0,0.0,NaN,0,2,TCMR,-,GSM2590943
1,GSM2590944,ABMR,0.0,1.0,3.0,2.0,3.0,2.0,0.0,1.0,1.0,1.0,1.0,NaN,0,4,TCMR,-,GSM2590944
2,GSM2590945,NR,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,1.0,1,1,Bord.,NR,GSM2590945
3,GSM2590946,ABMR,1.0,1.0,0.0,2.0,0.0,2.0,0.0,2.0,3.0,1.0,0.0,NaN,0,5,ABMR,-,GSM2590946
4,GSM2590947,NR,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,1,1,AKI,NR,GSM2590947


In [2]:
print("=== Overall class distribution ===")
print(clean_df['mirage_label'].value_counts())
print()
print(clean_df['mirage_label'].value_counts(normalize=True).round(3) * 100)

print("\n=== Split group sizes ===")
group_sizes = clean_df['split_group'].value_counts()
print(group_sizes.value_counts())  # should show: 1202 groups of size 1, 3 groups of size 2

print("\n=== Class distribution among grouped (non-singleton) pairs ===")
grouped_gsms = clean_df[clean_df['split_group'].isin(group_sizes[group_sizes > 1].index)]
print(grouped_gsms[['GSM', 'mirage_label', 'split_group']])

=== Overall class distribution ===
mirage_label
NR       774
ABMR     326
TCMR      81
Mixed     27
Name: count, dtype: int64

mirage_label
NR       64.1
ABMR     27.0
TCMR      6.7
Mixed     2.2
Name: proportion, dtype: float64

=== Split group sizes ===
count
1    1202
2       3
Name: count, dtype: int64

=== Class distribution among grouped (non-singleton) pairs ===
             GSM mirage_label     split_group
74    GSM2591017         ABMR  grp_GSM2591017
531   GSM2591474         ABMR  grp_GSM2591474
777   GSM2591726         ABMR  grp_GSM2591726
864   GSM2591813         ABMR  grp_GSM2591017
980   GSM2591929           NR  grp_GSM2591474
1094  GSM2592043         ABMR  grp_GSM2591726


In [3]:
from sklearn.model_selection import StratifiedGroupKFold

X_dummy = clean_df.index.values  # placeholder, StratifiedGroupKFold only needs y and groups to split
y = clean_df['mirage_label'].values
groups = clean_df['split_group'].values

sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)

clean_df['cv_fold'] = -1

for fold_idx, (train_idx, val_idx) in enumerate(sgkf.split(X_dummy, y, groups)):
    clean_df.loc[clean_df.index[val_idx], 'cv_fold'] = fold_idx

print("Fold assignment counts:")
print(clean_df['cv_fold'].value_counts().sort_index())

print("\n=== Class distribution per fold ===")
print(pd.crosstab(clean_df['cv_fold'], clean_df['mirage_label']))

print("\n=== Verify: no split_group spans multiple folds ===")
group_fold_counts = clean_df.groupby('split_group')['cv_fold'].nunique()
print("Groups spanning >1 fold (should be 0):", (group_fold_counts > 1).sum())

Fold assignment counts:
cv_fold
0    242
1    242
2    242
3    241
4    241
Name: count, dtype: int64

=== Class distribution per fold ===
mirage_label  ABMR  Mixed   NR  TCMR
cv_fold                             
0               69      7  153    13
1               59      6  156    21
2               60      2  157    23
3               65      7  156    13
4               73      5  152    11

=== Verify: no split_group spans multiple folds ===
Groups spanning >1 fold (should be 0): 0


In [4]:
best_seed = None
best_min_mixed = -1

for seed in range(20):
    sgkf_test = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=seed)
    fold_assign = np.full(len(clean_df), -1)
    for fold_idx, (train_idx, val_idx) in enumerate(sgkf_test.split(X_dummy, y, groups)):
        fold_assign[val_idx] = fold_idx

    temp_df = pd.DataFrame({'mirage_label': y, 'cv_fold': fold_assign})
    mixed_per_fold = temp_df[temp_df['mirage_label'] == 'Mixed']['cv_fold'].value_counts()
    min_mixed = mixed_per_fold.min() if len(mixed_per_fold) == 5 else 0

    if min_mixed > best_min_mixed:
        best_min_mixed = min_mixed
        best_seed = seed

print(f"Best seed: {best_seed}, minimum Mixed samples in any fold: {best_min_mixed}")

Best seed: 2, minimum Mixed samples in any fold: 5


In [5]:
sgkf_final = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=2)

clean_df['cv_fold'] = -1
for fold_idx, (train_idx, val_idx) in enumerate(sgkf_final.split(X_dummy, y, groups)):
    clean_df.loc[clean_df.index[val_idx], 'cv_fold'] = fold_idx

print("Fold assignment counts:")
print(clean_df['cv_fold'].value_counts().sort_index())

print("\n=== Class distribution per fold (final) ===")
print(pd.crosstab(clean_df['cv_fold'], clean_df['mirage_label']))

print("\n=== Verify: no split_group spans multiple folds ===")
group_fold_counts = clean_df.groupby('split_group')['cv_fold'].nunique()
print("Groups spanning >1 fold (should be 0):", (group_fold_counts > 1).sum())

# Save the finalized master table with locked CV folds
final_master_path = f'{DATA_DIR}/gse98320_master_pathology_labels.csv'
clean_df.to_csv(final_master_path, index=False)
print(f"\nSaved finalized master table (with cv_fold) to {final_master_path}")

Fold assignment counts:
cv_fold
0    242
1    242
2    241
3    241
4    242
Name: count, dtype: int64

=== Class distribution per fold (final) ===
mirage_label  ABMR  Mixed   NR  TCMR
cv_fold                             
0               63      6  160    13
1               69      5  154    14
2               60      5  159    17
3               59      5  156    21
4               75      6  145    16

=== Verify: no split_group spans multiple folds ===
Groups spanning >1 fold (should be 0): 0

Saved finalized master table (with cv_fold) to /content/drive/MyDrive/MIRAGE/data/raw/GSE98320/gse98320_master_pathology_labels.csv


In [6]:
from google.colab import files
files.download('/content/drive/MyDrive/MIRAGE/data/raw/GSE98320/gse98320_master_pathology_labels.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>